In [1]:
import json
import pandas as pd
import os
from tqdm import tqdm
from glob import glob
from datasets import Dataset, Audio

audio = Audio(sampling_rate = 16000)

In [2]:
with open('instructions-keys.json') as fopen:
    instructions = json.load(fopen)

In [3]:
with open('fix-instructions-mixtral-multiturn.json') as fopen:
    mixtral_multiturn = json.load(fopen)

In [4]:
files = glob('short-coding-*.json')
files = [f for f in files if os.path.exists(f.replace('.json', ''))]
files

['short-coding-2.json', 'short-coding-0.json', 'short-coding-1.json']

In [5]:
filtered = []
for f in files:
    folder = f.replace('.json', '')
    with open(f) as fopen:
        data = json.load(fopen)
    
    for i, d in tqdm(enumerate(data)):
        filename = os.path.join(folder, f'{i}.mp3')
        if not os.path.exists(filename):
            continue
        
        d = {
            'prompt': json.dumps([
                {"role": "user", "content": [{"type": "audio", "audio_url": "audio.wav"}]},
                {'role': 'assistant', 'content': d['answer']},
            ]),
            'question': d['question'],
            'audio_filename': filename,
            'dataset': 'short-coding',
            'speaker': d['speaker']
        }
        filtered.append(d)
len(filtered)

18990it [00:00, 107255.60it/s]
18991it [00:00, 143799.56it/s]
18991it [00:00, 101597.58it/s]


45415

In [6]:
files = glob('partition-instructions-part-*.json')
files = [f for f in files if os.path.exists(f.replace('.json', ''))]
files

['partition-instructions-part-7.json',
 'partition-instructions-part-16.json',
 'partition-instructions-part-8.json',
 'partition-instructions-part-0.json',
 'partition-instructions-part-2.json',
 'partition-instructions-part-15.json',
 'partition-instructions-part-4.json',
 'partition-instructions-part-5.json',
 'partition-instructions-part-6.json',
 'partition-instructions-part-9.json',
 'partition-instructions-part-14.json',
 'partition-instructions-part-10.json',
 'partition-instructions-part-3.json',
 'partition-instructions-part-11.json',
 'partition-instructions-part-1.json',
 'partition-instructions-part-13.json',
 'partition-instructions-part-17.json',
 'partition-instructions-part-12.json']

In [7]:
from collections import defaultdict

selected = []
already = defaultdict(set)
count = defaultdict(int)
for f in files:
    folder = f.replace('.json', '')
    with open(f) as fopen:
        data = json.load(fopen)
        
    for i, d in tqdm(enumerate(data)):
        filename = os.path.join(folder, f'{i}.mp3')
        if not os.path.exists(filename):
            continue
        
        d['prompt'] = json.dumps(d['prompt'])
        d['audio_filename'] = filename
        d['dataset'] = instructions.get(d['prompt'], 'unknown')
        if 'mixtral' in d['dataset'] and d['question'] in mixtral_multiturn:
            d['prompt'] = mixtral_multiturn[d['question']]
        
        count[d['dataset']] += 1
        
        q = d['question'].lower()
        if q in already[d['dataset']]:
            continue
        
        already[d['dataset']].add(q)
        filtered.append(d)
        
len(filtered)

30000it [00:00, 95881.71it/s]
30000it [00:00, 66643.18it/s]
30000it [00:00, 90548.98it/s]
30000it [00:00, 122168.58it/s]
30000it [00:00, 51657.36it/s]
30000it [00:00, 69356.50it/s] 
30000it [00:00, 84341.19it/s]
30000it [00:00, 146769.11it/s]
30000it [00:00, 178442.04it/s]
30000it [00:00, 89725.28it/s]
30000it [00:00, 93723.83it/s] 
30000it [00:00, 91206.23it/s]
30000it [00:00, 62178.52it/s]
30000it [00:00, 91620.70it/s]
30000it [00:00, 110917.48it/s]
30000it [00:00, 116390.76it/s]
30000it [00:01, 28044.21it/s]
30000it [00:00, 87323.85it/s]


464971

In [8]:
for k, v in already.items():
    print(k, len(v), count[k])

mixtral_critis_malaysia 88792 88831
unknown 489 495
force_jawi 23660 63930
mixtral_critis_politician 91244 91340
chatgpt4_malaysian_general_qa 26505 26573
malaysian_ultrachat 87266 89992
malaysian_alpaca 18243 18245
synthetic_coding 3282 3282
mixtral_conversation_stupid 43059 43506
mixtral_factually_wrong 35901 38166
force_tamil 40 40
force_mandarin 1075 1125


In [9]:
with open('tatabahasa.json') as fopen:
    tatabahasa = json.load(fopen)
    
for i, row in tqdm(enumerate(tatabahasa)):
    filename = os.path.join('tatabahasa', f'{i}.mp3')
    if not os.path.exists(filename):
        continue
    q = row['question']
    if 'IV' in q or 'II' in q:
        continue
    d = {
        'prompt': json.dumps([
            {"role": "user", "content": [{"type": "audio", "audio_url": "audio.wav"}]},
            {'role': 'assistant', 'content': row['answer']},
        ]),
        'question': row['question'],
        'audio_filename': filename,
        'dataset': 'tatabahasa',
        'speaker': row['speaker']
    }
    filtered.append(d)

1284it [00:00, 173061.03it/s]


In [10]:
filtered[-1]

{'prompt': '[{"role": "user", "content": [{"type": "audio", "audio_url": "audio.wav"}]}, {"role": "assistant", "content": "C. dengan, ke"}]',
 'question': 'Isi tempat kosong dalam ayat-ayat di bawah dengan jawapan yang paling sesuai.\nKami dijangka __________ terlewat sampai __________ Pulau Langkawi kerana kereta mengalami kerosakan.\n\nA. dari, di\nB. akan, di\nC. akan, dari\nD. dengan, ke',
 'audio_filename': 'tatabahasa/1283.mp3',
 'dataset': 'tatabahasa',
 'speaker': {'audio': 'dedup-parliament/parlimen-24k-LANGSUNG： Persidangan Dewan Rakyat ｜ Mesyuarat Pertama Penggal Ketiga 7 Mac 2024 ｜ Sesi Petang [RvL2ZIBGkzM]_000_278.mp3',
  'transcription': 'Malah, impact-nya turut disedari sendiri oleh yang amat berhormat Perdana Menteri yang menadakan lawatan ke negeri Pulau Pinang pada bulan yang lalu.'}}

In [11]:
with open('mallm.json') as fopen:
    mallm = json.load(fopen)
    
for i, row in tqdm(enumerate(mallm)):
    filename = os.path.join('mallm', f'{i}.mp3')
    if not os.path.exists(filename):
        continue
    q = row['question']
    if 'IV' in q or 'II' in q:
        continue
    d = {
        'prompt': json.dumps([
            {"role": "user", "content": [{"type": "audio", "audio_url": "audio.wav"}]},
            {'role': 'assistant', 'content': row['answer']},
        ]),
        'question': row['question'],
        'audio_filename': filename,
        'dataset': 'mallm',
        'speaker': row['speaker']
    }
    filtered.append(d)

6564it [00:00, 269126.89it/s]


In [12]:
len(filtered)

469477

In [13]:
dataset = Dataset.from_list(filtered)

In [14]:
dataset = dataset.cast_column("audio_filename", audio)
dataset

Dataset({
    features: ['prompt', 'question', 'audio_filename', 'dataset', 'speaker'],
    num_rows: 469477
})

In [15]:
dataset.push_to_hub('malaysia-ai/Speech-Instructions')

Uploading the dataset shards:   0%|          | 0/51 [00:00<?, ?it/s]

Map:   0%|          | 0/9206 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/93 [00:00<?, ?ba/s]

Map:   0%|          | 0/9206 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/93 [00:00<?, ?ba/s]

...

Creating parquet from Arrow format:   0%|          | 0/93 [00:00<?, ?ba/s]

Map:   0%|          | 0/9205 [00:00<?, ? examples/s]

Creating parquet from Arrow format:   0%|          | 0/93 [00:00<?, ?ba/s]

README.md:   0%|          | 0.00/1.10k [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Speech-Instructions/commit/ddb4e51a57b306770a28c553146e99a858cdc78a', commit_message='Upload dataset (part 00001-of-00002)', commit_description='', oid='ddb4e51a57b306770a28c553146e99a858cdc78a', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Speech-Instructions', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Speech-Instructions'), pr_revision=None, pr_num=None)